In [1]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

In [2]:
# 1. LOAD DATA INTO PANDAS (For easy manipulation)
housing = fetch_california_housing()
housingData = pd.DataFrame(housing.data, columns=housing.feature_names)
housingData['TargetPrice'] = housing.target

print(f"Original dataset size: {len(housingData)} houses")

# 2. REMOVE THE DATA WALL (The $500k outliers)
# The census capped prices at $500,001 (which is represented as 5.0 in this dataset).
# We completely drop these rows so the model doesn't learn a false "ceiling".
cleanData = housingData[housingData['TargetPrice'] < 5.0].copy()

outliersRemoved = len(housingData) - len(cleanData)
print(f"Cleaned dataset size: {len(cleanData)} houses (Removed {outliersRemoved} corrupted outliers)")

# 3. FEATURE ENGINEERING
# A. Bedroom Ratio: What percentage of the house is dedicated to bedrooms?
cleanData['BedRatio'] = cleanData['AveBedrms'] / cleanData['AveRooms']

# B. Wealth Density: Median income distributed over household occupancy
cleanData['WealthDensity'] = cleanData['MedInc'] / cleanData['AveOccup']

# 4. SPLIT THE NEW DATA
# array now has 10 columns instead of 8
xClean = cleanData.drop('TargetPrice', axis=1).values
yClean = cleanData['TargetPrice'].values

Original dataset size: 20640 houses
Cleaned dataset size: 19648 houses (Removed 992 corrupted outliers)


In [3]:
# Converting to numpy 
XTrain, XTest, YTrain, YTest = train_test_split(xClean, yClean, test_size=0.2, random_state=50) 

# FIX: axis=0 ensures we calculate mean/std for each feature column individually
xMean = np.mean(XTrain, axis=0)
xStd = np.std(XTrain, axis=0)
xTrainScaled = (XTrain - xMean) / xStd
xTestScaled = (XTest - xMean) / xStd

# Target Scaling
yMean = np.mean(YTrain)
yStd = np.std(YTrain)
yTrainScaled = (YTrain - yMean) / yStd
yTrainScaledMatrix = yTrainScaled.reshape(-1, 1)

In [11]:
class DenseLayer:

    # w : Weights
    # b : Biases
    def __init__(self, inSize, outSize, l2lambda=0.001):
        self.w = np.random.randn(inSize, outSize) * np.sqrt(2.0 / inSize)
        self.b = np.zeros((1, outSize))
        self.inputs = None

        # L2 Regularization
        self.l2lambda = l2lambda

        # ADAM optimizer
        self.wMomentum = np.zeros((inSize, outSize))
        self.wVelocity = np.zeros((inSize, outSize))
        self.bMomentum = np.zeros((1, outSize))
        self.bVelocity = np.zeros((1, outSize))
        self.timeStep = 0

    # w : Weights
    # b : Biases
    def forward(self, inputs):
        self.inputs = inputs
        return np.dot(inputs, self.w) + self.b

    # outGrad : Output Gradient
    # lr      : Learning Rate 
    def backward(self, outGrad, lr=0.01):
        wGrad = np.dot(self.inputs.T, outGrad)
        bGrad = np.sum(outGrad, axis=0, keepdims=True)
        inGrad = np.dot(outGrad, self.w.T)

        # Bias is not penalized as it does not contribute to overfitting
        wGrad += self.l2lambda * self.w

        self.timeStep += 1
        betaM = 0.9
        betaV = 0.999
        epsilon = 1e-8

        self.wMomentum = betaM * self.wMomentum + (1 - betaM) * wGrad
        self.bMomentum = betaM * self.bMomentum + (1 - betaM) * bGrad

        self.wVelocity = betaV * self.wVelocity + (1- betaV) * (wGrad ** 2)
        self.bVelocity = betaV * self.bVelocity + (1- betaV) * (bGrad ** 2)


        wMomentumHat = self.wMomentum / (1 - betaM ** self.timeStep)
        wVelocityHat = self.wVelocity / (1 - betaV ** self.timeStep)
        bMomentumHat = self.bMomentum / (1 - betaM ** self.timeStep)
        bVelocityHat = self.bVelocity / (1 - betaV ** self.timeStep)

        self.w -= lr * wMomentumHat / (np.sqrt(wVelocityHat) + epsilon)
        self.b -= lr * bMomentumHat / (np.sqrt(bVelocityHat) + epsilon)

        return inGrad        

In [20]:
class ReluLayer:
    
    def __init__(self):
        self.inputs = None

    def forward(self, inputs):
        self.inputs = inputs
        return np.where(inputs < 0, inputs * 0.01, inputs)

    def backward(self, outGrad, lr=0.01):
        return outGrad * np.where(self.inputs > 0, 1, 0.01)

In [6]:
class NeuralNetwork:

    def __init__(self, layerSizes):
        self.layers = []
        self._buildNetwork(layerSizes)            

    def _buildNetwork(self, layerSizes):
        for i in range(len(layerSizes) - 1):
            inSize = layerSizes[i]
            outSize = layerSizes[i + 1]

            self.layers.append(DenseLayer(inSize, outSize))

            if i < len(layerSizes) - 2:
                self.layers.append(ReluLayer())            
        
    def addLayer(self, layer):
        self.layers.append(layer)

    def predict(self, inputs):
        outputs = inputs
        for layer in self.layers:
            outputs = layer.forward(outputs)

        return outputs

    def train(self, xTrain, yTrain, epochs=100, lr=0.01, batchSize=64):
        nSamples = xTrain.shape[0]
        if hasattr(yTrain, 'values'):
            yTrain = yTrain.values
        yTrain = yTrain.reshape(-1, 1)

        for epoch in range(epochs):

            indices = np.arange(nSamples)
            np.random.shuffle(indices)

            xShuffled = xTrain[indices]
            yShuffled = yTrain[indices]

            for startIdx in range(0, nSamples, batchSize):
                endIdx = startIdx + batchSize
                xBatch = xShuffled[startIdx:endIdx]
                yBatch = yShuffled[startIdx:endIdx]

                actualBatchSize = xBatch.shape[0]

                prediction = self.predict(xBatch)
                error = prediction - yBatch
                grad = (2 / actualBatchSize) * error
            
                for layer in reversed(self.layers):
                    grad = layer.backward(grad, lr)

            if (epoch + 1) % 10 == 0:
                fullPrediction = self.predict(xTrain)
                mse = np.mean((fullPrediction - yTrain) ** 2)
                print(f"Epoch {epoch + 1:3} | Mini-Batch MSE: {mse:.4f}")

In [19]:
import pickle
with open("california_dnn.pkl", "rb") as file:
    model = pickle.load(file)

print("Model Loaded")

Model Loaded


In [21]:
architechture = [10, 128, 64, 16, 1]

model = NeuralNetwork(architechture)

model.train(xTrainScaled, yTrainScaled, epochs=300, lr=0.001, batchSize=64)

Epoch  10 | Mini-Batch MSE: 0.2416
Epoch  20 | Mini-Batch MSE: 0.2315
Epoch  30 | Mini-Batch MSE: 0.2097
Epoch  40 | Mini-Batch MSE: 0.2091
Epoch  50 | Mini-Batch MSE: 0.2109
Epoch  60 | Mini-Batch MSE: 0.2062
Epoch  70 | Mini-Batch MSE: 0.1998
Epoch  80 | Mini-Batch MSE: 0.1915
Epoch  90 | Mini-Batch MSE: 0.1958
Epoch 100 | Mini-Batch MSE: 0.1918
Epoch 110 | Mini-Batch MSE: 0.1900
Epoch 120 | Mini-Batch MSE: 0.1954
Epoch 130 | Mini-Batch MSE: 0.1830
Epoch 140 | Mini-Batch MSE: 0.1872
Epoch 150 | Mini-Batch MSE: 0.1844
Epoch 160 | Mini-Batch MSE: 0.1779
Epoch 170 | Mini-Batch MSE: 0.1931
Epoch 180 | Mini-Batch MSE: 0.1817
Epoch 190 | Mini-Batch MSE: 0.1720
Epoch 200 | Mini-Batch MSE: 0.1732
Epoch 210 | Mini-Batch MSE: 0.1717
Epoch 220 | Mini-Batch MSE: 0.1706
Epoch 230 | Mini-Batch MSE: 0.1718
Epoch 240 | Mini-Batch MSE: 0.1729
Epoch 250 | Mini-Batch MSE: 0.1632
Epoch 260 | Mini-Batch MSE: 0.1679
Epoch 270 | Mini-Batch MSE: 0.1731
Epoch 280 | Mini-Batch MSE: 0.1686
Epoch 290 | Mini-Bat

In [24]:
# --- EVALUATION ON TEST SET ---
# 3. EVALUATION
testPredictionsScaled = model.predict(xTestScaled)
testPredictions = (testPredictionsScaled * yStd) + yMean

yTestMatrix = YTest.reshape(-1, 1)
ssTotal = np.sum((yTestMatrix - np.mean(yTestMatrix)) ** 2)
ssResidual = np.sum((testPredictions - yTestMatrix) ** 2)
testR2 = 1 - (ssResidual / ssTotal)

print(f"R-squared (R2) Score:          {testR2:.4f}")

R-squared (R2) Score:          0.7728


In [23]:
# 1. Scale the test targets using the EXACT SAME mean and std from the training set
yTestScaled = (YTest - yMean) / yStd
yTestScaledMatrix = yTestScaled.reshape(-1, 1)

# 2. Initialize the heavy-duty architecture (10 input features!)
architecture = [10, 256, 128, 64, 1]
model = NeuralNetwork(architecture)

print("Training Regularized Network & Tracking Validation Loss...")

epochs = 150
batchSize = 64
nSamples = xTrainScaled.shape[0]

for epoch in range(epochs):
    # Shuffle training data at the start of every epoch
    indices = np.arange(nSamples)
    np.random.shuffle(indices)
    xShuffled = xTrainScaled[indices]
    yShuffled = yTrainScaledMatrix[indices]
    
    # Mini-batch training loop
    for startIdx in range(0, nSamples, batchSize):
        endIdx = startIdx + batchSize
        xBatch = xShuffled[startIdx:endIdx]
        yBatch = yShuffled[startIdx:endIdx]
        actualBatchSize = xBatch.shape[0]

        # Forward Pass
        prediction = model.predict(xBatch)
        error = prediction - yBatch
        
        # Backward Pass (Gradient Calculation)
        grad = (2 / actualBatchSize) * error
        for layer in reversed(model.layers):
            # Your L2 Regularization automatically applies inside here!
            grad = layer.backward(grad, lr=0.001)

    # Track Train vs Test Loss every 50 epochs
    if (epoch + 1) % 50 == 0:
        # Calculate Training Loss (MSE on scaled train data)
        trainPred = model.predict(xTrainScaled)
        trainLoss = np.mean((trainPred - yTrainScaledMatrix) ** 2)
        
        # Calculate Test Loss (MSE on scaled test data)
        testPred = model.predict(xTestScaled)
        testLoss = np.mean((testPred - yTestScaledMatrix) ** 2)
        
        print(f"Epoch {epoch + 1:3} | Train Loss: {trainLoss:.4f} | Test Loss: {testLoss:.4f}")

print("\nTraining Complete!")

Training Regularized Network & Tracking Validation Loss...
Epoch  50 | Train Loss: 0.2075 | Test Loss: 0.2371
Epoch 100 | Train Loss: 0.1976 | Test Loss: 0.2403
Epoch 150 | Train Loss: 0.1806 | Test Loss: 0.2285

Training Complete!


In [17]:
with open('california_dnn.pkl', 'wb') as file:
    pickle.dump(model, file)
print("Model saved")

Model saved
